In [1]:
import os
import sys
from pathlib import Path
# script_path = Path(__file__).resolve()
# project_root = script_path.parents[1]
# sys.path.insert(0, str(project_root))
import h5py
import numpy as np
from pathlib import Path
from diffractom import SinglePhaseForwardOperator
from diffractom.operators.single_phase_forward_operator import estimate_L_power
import yaml
from diffractom import FISTAHuber
import pyopencl.array as clarray
from diffractom import Grid
from diffractom import Material
import matplotlib.pyplot as plt
from orix import plot
from orix.quaternion import Orientation, symmetry
from orix.vector import Vector3d
from scipy.spatial.transform import Rotation as R
from sklearn.neighbors import KDTree

from integration.integrated_data import IntegratedData


In [2]:
ROOT = "/dtu/3d-imaging-center/projects/2025_QIM_BlackBeauty/raw_data_extern/2025_Danmax_Al1050"
PROCESS = os.path.join(ROOT, "process")
from integration.frame_loader import OUTPUT_PATH as INTEGRATED_PATH  # polarization-corrected run (single constant)
CIF_PATH = "/dtu/3d-imaging-center/projects/2025_QIM_BlackBeauty/raw_data_extern/2025_Danmax_Al1050/process/Aluminum"

In [3]:
niter = 200
niter_L_estimate = 10
N_eta = 360
N_Omega = 500
N_theta = 14
Nx = 108
Ny = 108
My = 148
cor_offset = 2
wavelength = 0.3559394165671113


In [4]:

# ---------------------------
# LOAD DATA
# ---------------------------

data = IntegratedData(INTEGRATED_PATH)  # no suffix
# ring normalisation below must never run on polarization-uncorrected data
assert data.polarization_corrected is True, (
    f"{data.json_path}: polarization_corrected={data.polarization_corrected} "
    "(None = not recorded, i.e. an older integration). The data must be reintegrated with the "
    "polarization correction (integration/01_inspect_integrate.ipynb) before binning/normalisation."
)
#print(data)               # summary: shape, dty/omega ranges, and each ring's q and hkl families

data.I                    # read-only np.memmap, (n_translations, n_omega, n_eta, n_rings)
data.q_nm                 # ring centres
data.hkl                  # per ring, a list of hkl families (a ring can hold several)
data.translations         # dty positions
data.omega_deg            # shared omega sequence
data.eta_deg             # azimuthal bin centres 



array([-179.5, -178.5, -177.5, -176.5, -175.5, -174.5, -173.5, -172.5,
       -171.5, -170.5, -169.5, -168.5, -167.5, -166.5, -165.5, -164.5,
       -163.5, -162.5, -161.5, -160.5, -159.5, -158.5, -157.5, -156.5,
       -155.5, -154.5, -153.5, -152.5, -151.5, -150.5, -149.5, -148.5,
       -147.5, -146.5, -145.5, -144.5, -143.5, -142.5, -141.5, -140.5,
       -139.5, -138.5, -137.5, -136.5, -135.5, -134.5, -133.5, -132.5,
       -131.5, -130.5, -129.5, -128.5, -127.5, -126.5, -125.5, -124.5,
       -123.5, -122.5, -121.5, -120.5, -119.5, -118.5, -117.5, -116.5,
       -115.5, -114.5, -113.5, -112.5, -111.5, -110.5, -109.5, -108.5,
       -107.5, -106.5, -105.5, -104.5, -103.5, -102.5, -101.5, -100.5,
        -99.5,  -98.5,  -97.5,  -96.5,  -95.5,  -94.5,  -93.5,  -92.5,
        -91.5,  -90.5,  -89.5,  -88.5,  -87.5,  -86.5,  -85.5,  -84.5,
        -83.5,  -82.5,  -81.5,  -80.5,  -79.5,  -78.5,  -77.5,  -76.5,
        -75.5,  -74.5,  -73.5,  -72.5,  -71.5,  -70.5,  -69.5,  -68.5,
      

In [7]:
import time
import numpy as np

# ---------------------------
# BINNING SETTINGS
# ---------------------------
N_OMEGA_BINS = 360
OMEGA_RANGE = 360.0          # angular span covered by the bins (deg)
OMEGA_START = None           # None -> omega.min()
DUPLICATE_MODE = "mean"      # overlapping dty measurements: "mean" or "first"

omega = np.asarray(data.omega_deg, dtype=float)
dty = np.asarray(data.translations, dtype=float)
n_trans, n_omega, n_eta, n_rings = data.shape

# ---------------------------
# OMEGA: averaging matrix
# ---------------------------
start = omega.min() if OMEGA_START is None else OMEGA_START
bin_w = OMEGA_RANGE / N_OMEGA_BINS
b = np.floor((omega - start) / bin_w).astype(int)
valid = (b >= 0) & (b < N_OMEGA_BINS)
counts_om = np.bincount(b[valid], minlength=N_OMEGA_BINS)
assert (counts_om > 0).all(), f"{(counts_om == 0).sum()} empty omega bins -- gap in the rotation?"

A = np.zeros((N_OMEGA_BINS, n_omega), dtype=np.float32)
A[b[valid], np.flatnonzero(valid)] = 1.0 / counts_om[b[valid]]   # each row averages the frames in that bin

omega_binned = A.astype(np.float64) @ omega          # mean omega of the frames in each bin
omega_edges = start + np.arange(N_OMEGA_BINS + 1) * bin_w
print(f"omega: {valid.sum()} of {n_omega} frames used ({(~valid).sum()} outside the bin range dropped), "
      f"{counts_om.min()}..{counts_om.max()} frames per bin")

# ---------------------------
# DTY: snap to a regular grid
# ---------------------------
u = np.unique(np.round(dty, 6))
step = np.median(np.diff(u))
n_y = int(round((dty.max() - dty.min()) / step)) + 1
gidx = np.round((dty - dty.min()) / step).astype(int)
resid = np.abs(dty - (dty.min() + gidx * step)).max() / step
mult = np.bincount(gidx, minlength=n_y)
assert resid < 0.05, f"translations are {resid:.2%} of a step off the grid -- snapping is not safe"
assert (mult > 0).all(), f"unmeasured grid points: {np.flatnonzero(mult == 0)}"

dty_grid = np.linspace(dty.min(), dty.max(), n_y)    # exactly linspaced, as the projector needs
sources = [np.flatnonzero(gidx == g).tolist() for g in range(n_y)]   # raw translation indices per grid point
print(f"dty:   {n_trans} measurements -> {n_y} grid points, step {step:.6f}; "
      f"{(mult > 1).sum()} points have duplicates ({DUPLICATE_MODE})")

# ---------------------------
# BINNING (in memory)
# ---------------------------
out_shape = (n_y, N_OMEGA_BINS, n_eta, n_rings)
I_binned = np.empty(out_shape, dtype=np.float32)
print(f"binned array: {out_shape}, {I_binned.nbytes / 1e9:.2f} GB in RAM")

def bin_omega(i_trans):
    X = np.asarray(data.I[i_trans], dtype=np.float32)              # (n_omega, n_eta, n_rings)
    return (A @ X.reshape(n_omega, -1)).reshape(N_OMEGA_BINS, n_eta, n_rings)

t0 = time.perf_counter()
for g in range(n_y):
    members = sources[g]
    if DUPLICATE_MODE == "first":
        members = members[:1]
    acc = bin_omega(members[0]).astype(np.float64)
    for m in members[1:]:
        acc += bin_omega(m)
    I_binned[g] = (acc / len(members)).astype(np.float32)

    if (g + 1) % 25 == 0 or g == n_y - 1:
        elapsed = time.perf_counter() - t0
        print(f"  {g + 1}/{n_y} grid points  (~{elapsed / (g + 1) * (n_y - g - 1) / 60:.1f} min remaining)")

# ---------------------------
# SANITY CHECK
# ---------------------------
# a grid point with a single measurement must conserve omega-summed intensity
g_chk = int(np.flatnonzero(mult == 1)[0])
raw_sum = np.asarray(data.I[sources[g_chk][0]], dtype=np.float64)[valid].sum()
binned_sum = (I_binned[g_chk].astype(np.float64) * counts_om[:, None, None]).sum()
rel = abs(raw_sum - binned_sum) / raw_sum
print(f"conservation check (grid point {g_chk}): rel. diff {rel:.2e}")
assert rel < 1e-4, "binning does not conserve intensity"

# ---------------------------
# WHAT THE REST OF THE NOTEBOOK USES
# ---------------------------
# I_binned       (n_y, N_OMEGA_BINS, n_eta, n_rings)  float32, in RAM
# dty_grid       (n_y,)          exactly linspaced
# omega_binned   (N_OMEGA_BINS,) mean omega of the frames in each bin
# counts_om      frames per omega bin
# mult           raw measurements per dty grid point (1 or 2)
# data.q_nm, data.hkl, data.eta_deg  unchanged

omega: 3001 of 3003 frames used (2 outside the bin range dropped), 8..9 frames per bin
dty:   181 measurements -> 151 grid points, step 0.010000; 30 points have duplicates (mean)
binned array: (151, 360, 360, 14), 1.10 GB in RAM
  25/151 grid points  (~1.8 min remaining)
  50/151 grid points  (~1.4 min remaining)
  75/151 grid points  (~1.0 min remaining)
  100/151 grid points  (~0.7 min remaining)
  125/151 grid points  (~0.4 min remaining)
  150/151 grid points  (~0.0 min remaining)
  151/151 grid points  (~0.0 min remaining)
conservation check (grid point 0): rel. diff 2.48e-09


In [ ]:

arr = arr/arr.sum(axis=(0,1,2), keepdims=True)*arr.sum()/500
all_data_reshaped1 = arr.reshape((3000, My, N_eta*N_theta))
all_data_reshaped = all_data_reshaped1[::3000//N_Omega]

for i in range(1,3000//N_Omega):
    all_data_reshaped += all_data_reshaped1[i::3000//N_Omega]

# ---------------------------
# MATERIAL
# ---------------------------
cif_folder = Path(cif_path)
filenames = [p.name for p in cif_folder.glob("*.cif")]
path = cif_folder / filenames[0]

material = Material.from_cif(
    cif_path=path,
    wavelength_kev=wavelength_kev,
    min_two_theta=min_two_theta,
    max_two_theta=max_two_theta,
    intensity_cutoff_fraction=reflection_cutoff,
)

# ---------------------------
# GRID
# ---------------------------
with h5py.File("/work3/msaca/basis_set.h5", "r") as f:
    numpy_orien = f["numpy_orien"][...][::15]
    scores = f["scores"][...]



def prune_rotations(Rmats, theta_deg, target=5000):
    # Convert to quaternions
    q = R.from_matrix(Rmats).as_quat()  # (x,y,z,w)
    q /= np.linalg.norm(q, axis=1, keepdims=True)

    # Antipodal equivalence: q and -q represent same rotation
    q_full = np.vstack([q, -q])

    tree = KDTree(q_full)

    theta = np.deg2rad(theta_deg)

    used = np.zeros(len(q_full), dtype=bool)
    keep = []

    for i in range(len(q)):
        if used[i]:
            continue

        keep.append(i)

        # mark neighbors as used
        idx = tree.query_radius(q[i:i+1], r=theta)[0]
        used[idx] = True

        if len(keep) >= target:
            break

    return Rmats[keep]


pruned_orient = prune_rotations(numpy_orien, theta_deg=0.15, target=50000)

pruned_orient = np.load('/work3/msaca/basis_3/numpy_matrix_flat_thin_2mrad.npy')
pruned_orient = pruned_orient.transpose((0,2,1))

def rotate_orientations_about_z(orientations: np.ndarray, degrees: float) -> np.ndarray:
    orientations = np.asarray(orientations, dtype=np.float64)

    if orientations.ndim != 3 or orientations.shape[1:] != (3, 3):
        raise ValueError(
            f"`orientations` must have shape (N, 3, 3), got {orientations.shape}"
        )

    theta = np.deg2rad(degrees)
    c = np.cos(theta)
    s = np.sin(theta)

    Rz = np.array([
        [c, -s, 0.0],
        [s,  c, 0.0],
        [0.0, 0.0, 1.0],
    ], dtype=np.float64)

    return Rz[None, :, :] @ orientations

pruned_orient = rotate_orientations_about_z(pruned_orient, 50.0)

sigma = 0.007
grid = Grid.from_rotation_matrices(pruned_orient, sigma=sigma)
print('Generated grid')
# grid_resolution_parameter = 64      # example
# kernel_sigma = 0.025               # example
# sigma_levels = [kernel_sigma]      # start with single level


# grid = Grid.from_hopf_fzone(
#     material.point_group_matrices,
#     grid_resolution_parameter=grid_resolution_parameter,
#     sigma_levels=sigma_levels,
# )


# ---------------------------
# OPERATOR
# ---------------------------
op = SinglePhaseForwardOperator(cfg=cfg, material=material, grid=grid, max_gb=0.2,
                verbose=True, normalized=True)


queue = op.queue
Nx, Ny, My, K = op.Nx, op.Ny, op.My, op.K

x_gpu = clarray.zeros(queue, (Nx, Ny, K), dtype=np.float32, order="F")

out_cpu = np.ascontiguousarray(all_data_reshaped, dtype=np.float32)
out_gpu = clarray.to_device(queue, out_cpu)

# ---------------------------
# SOLVE
# ---------------------------
norm_sq = estimate_L_power(op, niter=niter_L_estimate, seed=0, eps=1e-30, verbose=1)
print('Estimated operator norm')
solver = FISTAHuber(op, prox_kind="nonneg", lam=2.5e5, L=1.1*norm_sq, huber_delta=4000.0)

solver.run(x_gpu, out_gpu, niter=niter, verbose=1, diagnostics_interval=1)

prediction = op.direct(x_gpu)
prediction_cpu = prediction.get().reshape((N_Omega, My, N_eta, N_theta))


coeffs = x_gpu.get().transpose((0,1,2))[::-1,::-1]

# ---------------------------
# SAVE RECONSTRUCTION
# ---------------------------
reconstruction_path = run_dir / "reconstruction.h5"
with h5py.File(reconstruction_path, "w") as f:
    dset = f.create_dataset("x", data=coeffs, compression=None)
    f.create_dataset("orientations", data = pruned_orient, compression=None)
    dset.attrs["units"] = "Arbitrary units"
    f.attrs["sigma"] = sigma
    f.attrs["sigma_unit"] = 'Radians'
    f.attrs['Regularization+constraints'] = 'Nonneg'
    f.attrs['Optimizer'] = 'FISTA'
    f.attrs['N_iter'] = niter
    f.attrs['Datafit'] = 'Huber loss, delta = 4000'
    f.attrs['Normalized_rings'] = True
    f.attrs['N_eta'] = N_eta
    f.attrs['N_Omega'] = N_Omega




    #f.create_dataset("y", data=prediction_cpu, compression=None)

# ---------------------------
# PLOTS
# ---------------------------

plt.figure(figsize=(8,6))
plt.imshow(prediction_cpu[0,90], aspect='auto')
plt.colorbar()
plt.tight_layout()
plt.savefig(run_dir/"prediction_00.png", dpi=300)
plt.close()

plt.figure(figsize=(8,6))
plt.imshow(arr[0,90], aspect='auto')
plt.colorbar()
plt.tight_layout()
plt.savefig(run_dir/"measured_00.png", dpi=300)
plt.close()

plt.figure(figsize=(8,6))
plt.imshow(prediction_cpu[200,70], aspect='auto')
plt.colorbar()
plt.tight_layout()
plt.savefig(run_dir/"prediction_01.png", dpi=300)
plt.close()

plt.figure(figsize=(8,6))
plt.imshow(arr[200,70], aspect='auto')
plt.colorbar()
plt.tight_layout()
plt.savefig(run_dir/"measured_01.png", dpi=300)
plt.close()

plt.figure(figsize=(8,6))
plt.imshow(prediction_cpu[:,:].sum(axis=(0,1)), aspect='auto')
plt.colorbar()
plt.tight_layout()
plt.savefig(run_dir/"prediction_sum_00.png", dpi=300)
plt.close()

plt.figure(figsize=(8,6))
plt.imshow(arr[:,:].sum(axis=(0,1)), aspect='auto')
plt.colorbar()
plt.tight_layout()
plt.savefig(run_dir/"measured_sum_00.png", dpi=300)
plt.close()

plt.figure(figsize=(8,6))
plt.imshow(prediction_cpu[0,:].sum(axis=(0)), aspect='auto')
plt.colorbar()
plt.tight_layout()
plt.savefig(run_dir/"prediction_sumtrans_00.png", dpi=300)
plt.close()

plt.figure(figsize=(8,6))
plt.imshow(arr[0,:].sum(axis=(0)), aspect='auto')
plt.colorbar()
plt.tight_layout()
plt.savefig(run_dir/"measured_sumtrans_00.png", dpi=300)
plt.close()

plt.figure(figsize=(7,6))
plt.imshow(coeffs.sum(axis=-1))
plt.colorbar()
plt.tight_layout()
plt.savefig(run_dir/"coeff_sum.png", dpi=300)
plt.close()

grid_sp = R.concatenate(grid.rotations_at_level(0))

stepsize = 0.004
imshow_opts = {'extent':(0, coeffs.shape[0]*stepsize, coeffs.shape[1]*stepsize, 0)}

plt.figure(figsize=(7,6))
plt.imshow(coeffs.sum(axis=2), **imshow_opts)
plt.colorbar()
plt.tight_layout()
plt.savefig(run_dir/"coeff_sum_spatial.png", dpi=300)
plt.close()

sums = coeffs.sum(axis=-1)

def save_ipf(direction, name):
    ipfkey = plot.IPFColorKeyTSL(symmetry.Oh, direction=Vector3d(direction))
    orientations = Orientation.from_scipy_rotation(grid_sp)
    orientations.symmetry = ipfkey.symmetry
    rgb = ipfkey.orientation2color(orientations)[np.argmax(coeffs, axis=-1)]
    mask = coeffs.sum(axis=2)>0
    rgb[~mask] *= 0

    plt.figure(figsize=(7,6))
    plt.imshow(rgb[::-1, ::-1], **imshow_opts)
    plt.tight_layout()
    plt.savefig(run_dir/name, dpi=300)
    plt.close()

save_ipf([1,0,0], "ipf_x.png")
save_ipf([0,1,0], "ipf_y.png")
save_ipf([0,0,1], "ipf_z.png")

print("All outputs saved in:", run_dir)
